Imports

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import kagglehub
from umap import UMAP
from kagglehub import KaggleDatasetAdapter
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from langchain_google_genai import ChatGoogleGenerativeAI
from bertopic.representation import LangChain
from sklearn.model_selection import train_test_split

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Data

In [2]:
df = pd.read_csv('Data/combined_timeline_data.csv')
df.head()

,Date,Source,Summary,Associated Link Title,Associated Link URL,Theme
0,"January 4, 2020",WHO Announces Pneumonia Cases of Unknown Cause,The World Health Organization (WHO) announces ...,WHO on Twitter,https://fraser.stlouisfed.org/docs/historical/...,Pandemic
1,"January 8, 2020",CDC Issues Health Advisory,The Centers for Disease Control and Prevention...,CDC Health Advisory,https://fraser.stlouisfed.org/archival-collect...,Pandemic
2,"January 9, 2020",CDC Notes Appearance of Novel Coronavirus Outb...,The CDC notes the appearance of a novel corona...,CDC Report,https://fraser.stlouisfed.org/archival-collect...,Pandemic
3,"January 17, 2020, 2:00 pm EST",CDC Announces Enhanced Screenings for Those Tr...,The CDC and the Department of Homeland Securit...,CDC Press Release,https://fraser.stlouisfed.org/archival-collect...,Pandemic
4,"January 21, 2020",Washington State Department of Health Announce...,The Washington State Department of Health anno...,Snohomish Health District Media Release,https://fraser.stlouisfed.org/title/state-mate...,Pandemic


In [3]:
docs = df["Summary"].astype(str).tolist()
# Extract only the date part (remove everything after the year)
timestamps = pd.to_datetime(
    df["Date"].astype(str).str.extract(r'([A-Za-z]+\s+\d{1,2},\s+\d{4})')[0],
    errors="coerce"
).dt.date

# Keep valid rows only
data = pd.DataFrame({"doc": docs, "timestamp": timestamps}).dropna()
docs = data["doc"].tolist()
timestamps = data["timestamp"].tolist()

# Train/validation/test split: 70/15/15
train_docs, temp_docs, train_timestamps, temp_timestamps = train_test_split(
    docs,
    timestamps,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_docs, test_docs, val_timestamps, test_timestamps = train_test_split(
    temp_docs,
    temp_timestamps,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print(f"Train size: {len(train_docs)}")
print(f"Validation size: {len(val_docs)}")
print(f"Test size: {len(test_docs)}")

Train size: 364
Validation size: 78
Test size: 78


Prepare Models

In [4]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
train_embeddings = sentence_model.encode(train_docs, show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 997.59it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 12/12 [00:02<00:00,  5.69it/s]


Train BERTopic

In [5]:
print("Training BERTopic model on training split...\n")

umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words="english")

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  calculate_probabilities=True,
  verbose=True
)

# Train BERTopic only on train split
topics_train, probs_train = topic_model.fit_transform(train_docs, embeddings=train_embeddings)

print("\nModel training complete!")

2026-03-25 00:13:31,038 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model on training split...



2026-03-25 00:13:38,166 - BERTopic - Dimensionality - Completed ✓
2026-03-25 00:13:38,167 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-25 00:13:38,167 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-25 00:13:38,191 - BERTopic - Cluster - Completed ✓
2026-03-25 00:13:38,195 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-25 00:13:38,191 - BERTopic - Cluster - Completed ✓
2026-03-25 00:13:38,195 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-25 00:13:38,219 - BERTopic - Representation - Completed ✓
2026-03-25 00:13:38,219 - BERTopic - Representation - Completed ✓



Model training complete!


Get Topic Information

In [6]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,69,-1_aig_federal_billion_new,"[aig, federal, billion, new, reserve, york, ba...",[The Federal Reserve Board and the U.S. Treasu...
1,0,118,0_reserve_federal_facility_board,"[reserve, federal, facility, board, credit, lo...",[Treasury Secretary Mnuchin announces that he ...
2,1,59,1_covid_19_cdc_united,"[covid, 19, cdc, united, states, reports, vacc...",[The CDC reports their estimate that over 5 mi...
3,2,30,2_stock_preferred_purchases_purchase,"[stock, preferred, purchases, purchase, total,...",[The U.S. Treasury Department purchases a tota...
4,3,22,3_commission_congressional_oversight_developments,"[commission, congressional, oversight, develop...",[The Congressional Oversight Commission releas...
5,4,21,4_rate_percent_votes_points,"[rate, percent, votes, points, basis, target, ...",[The FOMC votes to reduce its target for the f...
6,5,16,5_mae_fannie_loss_net,"[mae, fannie, loss, net, billion, freddie, mac...",[Fannie Mae reports a loss of $23.2 billion fo...
7,6,15,6_fdic_indymac_guarantee_deposit,"[fdic, indymac, guarantee, deposit, insured, i...",[The Board of Directors of the Federal Deposit...
8,7,14,7_tarp_troubled_relief_asset,"[tarp, troubled, relief, asset, report, progra...",[The Office of the Special Inspector General f...


Visualize Topics

In [7]:
topic_model.visualize_documents(train_docs, embeddings=train_embeddings)

Visualize Topic Similarity

In [8]:
fig = topic_model.visualize_heatmap()
fig.show()

Hierarchichal Topic Visualization

In [9]:
hierarchical_topics = topic_model.hierarchical_topics(train_docs)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

print("💡 This shows the hierarchical structure of topics")

100%|██████████| 7/7 [00:00<00:00, 475.74it/s]



💡 This shows the hierarchical structure of topics


Visualize Topics over Time

In [10]:
topics_over_time = topic_model.topics_over_time(train_docs, train_timestamps)
topic_model.visualize_topics_over_time(topics_over_time)

2026-03-25 00:13:39,080 - BERTopic - WARNING: There are more than 100 unique timestamps (i.e., 305) which significantly slows down the application. Consider setting `nr_bins` to a value lower than 100 to speed up calculation. 
305it [00:00, 317.86it/s]



Preparing Validation

In [11]:
bertopic_topics = [
    [word for word, _ in topic_model.get_topic(t)] 
    for t in topic_model.get_topics().keys() if t != -1
]

# Project validation/test documents into trained topic space
val_topics, val_probs = topic_model.transform(val_docs)
test_topics, test_probs = topic_model.transform(test_docs)

print("Validation/Test transform complete")

Batches: 100%|██████████| 3/3 [00:00<00:00,  4.52it/s]
2026-03-25 00:13:40,753 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
Batches: 100%|██████████| 3/3 [00:00<00:00,  4.52it/s]
2026-03-25 00:13:40,753 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-25 00:13:43,614 - BERTopic - Dimensionality - Completed ✓
2026-03-25 00:13:43,615 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-25 00:13:43,618 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-03-25 00:13:43,625 - BERTopic - Probabilities - Completed ✓
2026-03-25 00:13:43,614 - BERTopic - Dimensionality - Completed ✓
2026-03-25 00:13:43,615 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-25 00:13:43,618 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-03-25 00:13:43,625 - BERTopic - Probabilities - Completed ✓
2026-03-25 00:13:43,625 - BERTopic -

Validation/Test transform complete


In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Preprocess documents for gensim
stop_words = set(stopwords.words('english'))

def preprocess_documents(doc_list):
    processed = []
    for doc in doc_list:
        tokens = word_tokenize(doc.lower())
        tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
        processed.append(tokens)
    return processed

processed_train_docs = preprocess_documents(train_docs)
processed_val_docs = preprocess_documents(val_docs)
processed_test_docs = preprocess_documents(test_docs)

# Create dictionary from train split only
id2word = Dictionary(processed_train_docs)

def calculate_metrics(topics, corpus_docs, dictionary):
    cm = CoherenceModel(topics=topics, texts=corpus_docs, dictionary=dictionary, coherence='c_v')
    coherence = cm.get_coherence()

    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words) if len(all_words) > 0 else 0

    return coherence, diversity

# Calculate metrics from train split topic representation
cv_bertopic, div_bertopic = calculate_metrics(bertopic_topics, processed_test_docs, id2word)

# Held-out diagnostics
val_outlier_ratio = float(np.mean(np.array(val_topics) == -1)) if len(val_topics) > 0 else np.nan
test_outlier_ratio = float(np.mean(np.array(test_topics) == -1)) if len(test_topics) > 0 else np.nan

print(f"BERTopic Coherence (C_v, train): {cv_bertopic:.4f}")
print(f"BERTopic Topic Diversity (train): {div_bertopic:.4f}")
print(f"Validation outlier ratio (-1 topics): {val_outlier_ratio:.4f}")
print(f"Test outlier ratio (-1 topics): {test_outlier_ratio:.4f}")

BERTopic Coherence (C_v, train): 0.7836
BERTopic Topic Diversity (train): 0.9000
Validation outlier ratio (-1 topics): 0.3205
Test outlier ratio (-1 topics): 0.3974
